# Day 7 — 合并技术面 + 基本面特征

**目标**: 将 Day4 技术特征 + Day6 基本面特征合并, 生成最终建模数据集

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

OUT_DIR = r"C:\Users\1\Desktop\项目\stock-data"
pd.set_option("display.max_columns", 100)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.sans-serif"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# ==========================================
# 第1步: 读取技术特征 (Day4)
# ==========================================
df_tech = pd.read_csv(os.path.join(OUT_DIR, "day4_features.csv"))
df_tech["Date"] = pd.to_datetime(df_tech["Date"])
print(f"技术特征: {df_tech.shape}")
print(f"日期: {df_tech['Date'].min().date()} ~ {df_tech['Date'].max().date()}")

# 技术特征从 v2_feature_list.txt 读取
tech_feature_path = os.path.join(OUT_DIR, "v2_feature_list.txt")
if os.path.exists(tech_feature_path):
    with open(tech_feature_path, "r", encoding="utf-8") as f:
        TECH_FEATURES = [l.strip() for l in f if l.strip()]
    print(f"技术特征列表: {len(TECH_FEATURES)} 个")
else:
    TECH_FEATURES = []

In [ ]:
# ==========================================
# 第2步: 读取基本面特征 (Day6) — 可选
# ==========================================
FUNDA_PATH = os.path.join(OUT_DIR, "day6_fundamental_features.csv")
HAS_FUNDAMENTALS = os.path.exists(FUNDA_PATH)

if HAS_FUNDAMENTALS:
    df_funda = pd.read_csv(FUNDA_PATH)
    df_funda["Date"] = pd.to_datetime(df_funda["Date"])
    print(f"基本面特征: {df_funda.shape}")
    print(f"日期: {df_funda['Date'].min().date()} ~ {df_funda['Date'].max().date()}")
else:
    print("WARNING: day6_fundamental_features.csv 不存在!")
    print("  请先运行 day5_fundamental_scraper.ipynb → day6_fundamental_features.ipynb")
    print("  本次仅使用技术面特征继续...")
    df_funda = None

In [ ]:
# ==========================================
# 第3步: 合并
# ==========================================
if HAS_FUNDAMENTALS and df_funda is not None:
    df = df_tech.merge(df_funda, on=["Date", "symbol"], how="left")
else:
    df = df_tech.copy()

print(f"合并后: {df.shape}")
print(f"\n各列缺失率:")
missing = df.isna().mean().sort_values(ascending=False)
for c, pct in missing.items():
    if pct > 0:
        print(f"  {c}: {pct:.1%}")

In [ ]:
# ==========================================
# 第4步: 处理缺失值
# ==========================================

# 基本面特征用forward-fill (同一股票内)
if HAS_FUNDAMENTALS and df_funda is not None:
    funda_cols = [c for c in df_funda.columns if c not in ["Date", "symbol"]]
    for c in funda_cols:
        if c in df.columns:
            df[c] = df.groupby("symbol")[c].fillna(method="ffill")

# 剩余NaN用-999标记 (树模型能处理)
df = df.fillna(-999)

print(f"处理后维度: {df.shape}")
print(f"剩余NaN: {df.isna().sum().sum()}")

In [ ]:
# ==========================================
# 第5步: 定义特征集
# ==========================================

LABEL_COL = "crash_binary"  # MDD <= -15% = crash
ID_COLS = ["Date", "symbol", "close", "return", "future_mdd_20",
           "label", "crash_binary"]

# 中间计算列 (不参与建模)
drop_cols = ID_COLS + ["ema_12", "ema_26", "ma_5", "ma_10", "ma_20", "ma_60"]

feature_cols = [c for c in df.columns if c not in drop_cols]

print(f"特征数: {len(feature_cols)}")
print("\n特征列表:")
for i, f in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {f}")

In [ ]:
# ==========================================
# 第6步: 特征相关性热力图 (前20个特征)
# ==========================================

# 取前20个特征做相关性
top20 = feature_cols[:20]
corr = df[top20 + [LABEL_COL]].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, square=True, linewidths=0.5, ax=ax,
            annot_kws={"size": 7})
ax.set_title("特征相关性矩阵 (Top 20 + Label)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "day7_corr_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ==========================================
# 第7步: 标签分布 & 时间序列切分点
# ==========================================

if LABEL_COL in df.columns:
    print(f"标签分布 ({LABEL_COL}):")
    print(df[LABEL_COL].value_counts())
    print(f"Crash比例: {df[LABEL_COL].mean():.2%}")
else:
    print(f"WARNING: {LABEL_COL} 列不存在!")

# 按时序划分: 训练(2015-2021) / 验证(2022-2023) / 测试(2024-2025)
train_end = pd.Timestamp("2021-12-31")
val_end = pd.Timestamp("2023-12-31")

train_mask = df["Date"] <= train_end
val_mask = (df["Date"] > train_end) & (df["Date"] <= val_end)
test_mask = df["Date"] > val_end

print(f"\n训练集 (<=2021): {train_mask.sum()} ({train_mask.mean():.0%})")
print(f"验证集 (2022-2023): {val_mask.sum()} ({val_mask.mean():.0%})")
print(f"测试集 (2024-2025): {test_mask.sum()} ({test_mask.mean():.0%})")

if LABEL_COL in df.columns:
    print(f"\n各集Crash比例:")
    print(f"  Train: {df.loc[train_mask, LABEL_COL].mean():.2%}")
    print(f"  Val:   {df.loc[val_mask, LABEL_COL].mean():.2%}")
    print(f"  Test:  {df.loc[test_mask, LABEL_COL].mean():.2%}")

In [ ]:
# ==========================================
# 第8步: 保存最终建模数据
# ==========================================

output_cols = ID_COLS + feature_cols

df[output_cols].to_csv(
    os.path.join(OUT_DIR, "day7_modeling_dataset.csv"),
    index=False, encoding="utf-8-sig"
)

# 也保存特征列表供后续使用
with open(os.path.join(OUT_DIR, "day7_feature_list.txt"), "w", encoding="utf-8") as f:
    for feat in feature_cols:
        f.write(feat + "\n")

print("Day7 完成!")
print(f"  建模数据集: day7_modeling_dataset.csv ({df[output_cols].shape})")
print(f"  特征列表: day7_feature_list.txt ({len(feature_cols)} 个特征)")